In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

In [3]:
# np.random.seed(6242)
data = pd.read_csv('pitcher_data_detailed_cleaned.csv', parse_dates = ['game_date'])
# data['month'] = data['game_date'].dt.month
# data['year'] = data['game_date'].dt.year
# data['day'] = data['game_date'].dt.day
data['period'] = data['game_date'].dt.to_period('M')
data = data.groupby(['player_name', 'period']).agg({'pitch_name': 'count', 
                                                    'release_speed': 'mean',
                                                    'release_spin_rate': 'mean',    
                                                    'release_extension': 'mean',
                                                    # 'pfx_x': 'mean',
                                                    # 'pfx_z': 'mean',
                                                    'spin_axis': 'mean',
                                                    'release_pos_x': 'mean',
                                                    'release_pos_z': 'mean'}).reset_index().sort_values(['player_name', 'period'])
data['spin_rate_shifted'] = data.groupby('player_name')['release_spin_rate'].shift(1)
data = data.dropna()


rfModel = RandomForestRegressor(random_state = 6242)
paramGrid = {
    'n_estimators': [50, 100, 300, 500, 1000],
    'max_depth': [5, 10, 20],
    'min_samples_split': [2, 5, 15]
}

gridCV = GridSearchCV(
    rfModel,
    param_grid = paramGrid,
    cv = TimeSeriesSplit(n_splits = 5),
    scoring = 'neg_mean_squared_error'
)
data.head()

,player_name,period,pitch_name,release_speed,release_spin_rate,release_extension,spin_axis,release_pos_x,release_pos_z,spin_rate_shifted
1,"Almonte, Yency",2018-07,61,93.260656,2232.786885,6.216393,187.295082,-2.329672,5.859672,2318.842105
2,"Almonte, Yency",2018-08,50,91.520000,2205.720000,6.058000,175.060000,-2.960600,5.945800,2232.786885
3,"Almonte, Yency",2018-09,107,91.800935,2255.037383,6.222430,168.009346,-2.673084,5.894299,2205.720000
4,"Almonte, Yency",2019-04,42,93.159524,2242.285714,6.233333,192.595238,-2.065476,5.841905,2255.037383
5,"Almonte, Yency",2019-05,111,91.835135,2202.000000,6.308108,174.972973,-1.969189,5.786847,2242.285714


In [4]:
xTrain = data.loc[data['period'] < '2022-08', :]
yTrain = xTrain['release_spin_rate']
nameDataTrain = xTrain.iloc[:, :3]
xTrain = xTrain.iloc[:, 2:].drop('release_spin_rate', axis = 1)

xTest = data.loc[data['period'] >= '2022-08', :]
yTest = xTest['release_spin_rate']
nameDataTest = xTest.iloc[:, :3]
spinRateTest = xTest['release_spin_rate']
xTest = xTest.iloc[:, 2:].drop('release_spin_rate', axis = 1)

gridCV.fit(xTrain, yTrain)
rfBestModel = gridCV.best_estimator_
rfPred = rfBestModel.predict(xTest)
rfErr = mean_squared_error(yTest, rfPred)
# print(f'Random Forest MSE: {rfErr}')


In [5]:
nameDataTest['predictions'] = rfPred
nameDataTest['actual_spin_rate'] = spinRateTest
nameDataTest['residual'] = nameDataTest['predictions'] - nameDataTest['actual_spin_rate']
nameDataTest['period'] = nameDataTest['period'].dt.to_timestamp()
nameDataTest.to_csv('random_forest_predictions.csv', index = False)

# print(gridCV.best_params_)
# print(xTest.shape[0], len(rfPred))